# Explainable RAG-Based Credit Card Recommendation System

**Full research pipeline**: 20 Indian credit cards · 2,000 users · 4 models · 4 scenarios · alpha ablation · statistical tests · case study · latency analysis

**Install**: `pip install sentence-transformers faiss-cpu scipy scikit-learn`

In [ ]:
# Install if needed:
# !pip install sentence-transformers faiss-cpu scipy scikit-learn -q

import numpy as np
import pandas as pd
import random, time, json, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy import stats
from sentence_transformers import SentenceTransformer
import faiss

np.random.seed(42)
random.seed(42)
print('Dependencies loaded successfully.')

## Phase 1 — Card Dataset (20 authentic Indian credit cards, 4 issuers)

In [ ]:
def create_card_dataset():
    cards = [
        # HDFC Bank
        {'card_name': 'HDFC MoneyBack+',    'bank': 'HDFC',  'annual_fee': 500,   'income_requirement': 300000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.5,
         'shopping_reward_rate': 3.3, 'online_reward_rate': 3.3,
         'lounge_access': 0,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 500},
        {'card_name': 'HDFC Millennia',     'bank': 'HDFC',  'annual_fee': 1000,  'income_requirement': 500000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 5.0, 'online_reward_rate': 5.0,
         'lounge_access': 8,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 1000},
        {'card_name': 'HDFC Regalia Gold',  'bank': 'HDFC',  'annual_fee': 2500,  'income_requirement': 800000,
         'travel_reward_rate': 1.3, 'dining_reward_rate': 1.3,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 1.3, 'online_reward_rate': 1.3,
         'lounge_access': 18, 'forex_markup': 2.0, 'fuel_waiver': 1, 'welcome_bonus_value': 2000},
        {'card_name': 'HDFC Diners Black',  'bank': 'HDFC',  'annual_fee': 10000, 'income_requirement': 1750000,
         'travel_reward_rate': 3.3, 'dining_reward_rate': 3.3,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 3.3, 'online_reward_rate': 3.3,
         'lounge_access': 36, 'forex_markup': 2.0, 'fuel_waiver': 1, 'welcome_bonus_value': 5000},
        {'card_name': 'HDFC Infinia Metal', 'bank': 'HDFC',  'annual_fee': 12500, 'income_requirement': 3000000,
         'travel_reward_rate': 3.3, 'dining_reward_rate': 3.3,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 3.3, 'online_reward_rate': 3.3,
         'lounge_access': 50, 'forex_markup': 2.0, 'fuel_waiver': 1, 'welcome_bonus_value': 6250},
        # ICICI Bank
        {'card_name': 'ICICI Amazon Pay',   'bank': 'ICICI', 'annual_fee': 0,     'income_requirement': 240000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 3.0, 'online_reward_rate': 5.0,
         'lounge_access': 4,  'forex_markup': 1.99,'fuel_waiver': 1, 'welcome_bonus_value': 0},
        {'card_name': 'ICICI Coral',        'bank': 'ICICI', 'annual_fee': 2000,  'income_requirement': 300000,
         'travel_reward_rate': 0.5, 'dining_reward_rate': 0.5,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 0.5, 'online_reward_rate': 0.5,
         'lounge_access': 4,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 2500},
        {'card_name': 'ICICI Sapphiro',     'bank': 'ICICI', 'annual_fee': 6500,  'income_requirement': 600000,
         'travel_reward_rate': 4.0, 'dining_reward_rate': 2.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 2.0, 'online_reward_rate': 2.0,
         'lounge_access': 6,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 13000},
        {'card_name': 'ICICI Rubyx',        'bank': 'ICICI', 'annual_fee': 3000,  'income_requirement': 900000,
         'travel_reward_rate': 4.0, 'dining_reward_rate': 2.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 2.0, 'online_reward_rate': 2.0,
         'lounge_access': 8,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 5000},
        {'card_name': 'ICICI Emeralde',     'bank': 'ICICI', 'annual_fee': 12000, 'income_requirement': 2500000,
         'travel_reward_rate': 4.0, 'dining_reward_rate': 4.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 4.0, 'online_reward_rate': 4.0,
         'lounge_access': 30, 'forex_markup': 1.5, 'fuel_waiver': 1, 'welcome_bonus_value': 12000},
        # SBI Cards
        {'card_name': 'SBI SimplyCLICK',    'bank': 'SBI',   'annual_fee': 999,   'income_requirement': 180000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 1.0, 'online_reward_rate': 5.0,
         'lounge_access': 0,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 500},
        {'card_name': 'SBI SimplySAVE',     'bank': 'SBI',   'annual_fee': 499,   'income_requirement': 150000,
         'travel_reward_rate': 0.5, 'dining_reward_rate': 4.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 4.0, 'online_reward_rate': 0.5,
         'lounge_access': 0,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 500},
        {'card_name': 'SBI Cashback',       'bank': 'SBI',   'annual_fee': 999,   'income_requirement': 200000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 1.0, 'online_reward_rate': 5.0,
         'lounge_access': 0,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 0},
        {'card_name': 'SBI PRIME',          'bank': 'SBI',   'annual_fee': 3000,  'income_requirement': 200000,
         'travel_reward_rate': 2.5, 'dining_reward_rate': 2.5,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 2.5, 'online_reward_rate': 2.5,
         'lounge_access': 12, 'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 3000},
        {'card_name': 'SBI Elite',          'bank': 'SBI',   'annual_fee': 10000, 'income_requirement': 1000000,
         'travel_reward_rate': 2.0, 'dining_reward_rate': 2.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 2.0, 'online_reward_rate': 2.0,
         'lounge_access': 14, 'forex_markup': 1.99,'fuel_waiver': 1, 'welcome_bonus_value': 5000},
        # Axis Bank
        {'card_name': 'Axis Neo',           'bank': 'Axis',  'annual_fee': 250,   'income_requirement': 150000,
         'travel_reward_rate': 0.5, 'dining_reward_rate': 1.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 1.0, 'online_reward_rate': 2.0,
         'lounge_access': 0,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 250},
        {'card_name': 'Axis MY ZONE',       'bank': 'Axis',  'annual_fee': 500,   'income_requirement': 250000,
         'travel_reward_rate': 1.0, 'dining_reward_rate': 4.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 4.0, 'online_reward_rate': 1.0,
         'lounge_access': 2,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 500},
        {'card_name': 'Axis Flipkart',      'bank': 'Axis',  'annual_fee': 500,   'income_requirement': 300000,
         'travel_reward_rate': 1.5, 'dining_reward_rate': 1.5,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 5.0, 'online_reward_rate': 4.0,
         'lounge_access': 4,  'forex_markup': 3.5, 'fuel_waiver': 1, 'welcome_bonus_value': 500},
        {'card_name': 'Axis Magnus',        'bank': 'Axis',  'annual_fee': 12500, 'income_requirement': 1800000,
         'travel_reward_rate': 3.0, 'dining_reward_rate': 3.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 3.0, 'online_reward_rate': 3.0,
         'lounge_access': 24, 'forex_markup': 2.0, 'fuel_waiver': 1, 'welcome_bonus_value': 12500},
        {'card_name': 'Axis Reserve',       'bank': 'Axis',  'annual_fee': 50000, 'income_requirement': 5000000,
         'travel_reward_rate': 4.0, 'dining_reward_rate': 4.0,  'fuel_reward_rate': 0.0,
         'shopping_reward_rate': 4.0, 'online_reward_rate': 4.0,
         'lounge_access': 100,'forex_markup': 1.5, 'fuel_waiver': 1, 'welcome_bonus_value': 50000},
    ]
    return pd.DataFrame(cards)

df_cards = create_card_dataset()
print(f'Cards: {len(df_cards)} | Issuers: {df_cards["bank"].nunique()}')
df_cards[['card_name','bank','annual_fee','income_requirement']]


## Phase 2 — Synthetic User Generation (2,000 users)

In [ ]:
def create_users(n_users=2000):
    np.random.seed(42)
    income = np.random.lognormal(13.8, 0.75, n_users)
    income = np.clip(income, 150000, 5000000)  # floor activates eligibility filter
    df = pd.DataFrame({'user_id': range(1, n_users+1), 'annual_income': income.astype(int)})
    ratio = np.random.uniform(0.2, 0.4, n_users)
    df['monthly_spend'] = (df['annual_income'] * ratio) / 12
    cats = ['travel','dining','fuel','shopping','online']
    dist = np.random.dirichlet([2]*5, n_users)
    for i, cat in enumerate(cats):
        df[f'{cat}_spend'] = df['monthly_spend'] * dist[:, i]
    return df

df_users = create_users(2000)
print(f'Users: {len(df_users):,} | Income range: Rs {df_users["annual_income"].min():,} - Rs {df_users["annual_income"].max():,}')
df_users.describe().round(0)


## Financial Utility Model + Oracle Builder

In [ ]:
def calculate_net_benefit(user, card):
    cats = ['travel','dining','fuel','shopping','online']
    reward = sum(user[f'{c}_spend'] * 12 * (card[f'{c}_reward_rate'] / 100) for c in cats)
    return reward + card['welcome_bonus_value'] - card['annual_fee']

def build_oracle(df_users, df_cards, noise=0.0, seed=42):
    rng = random.Random(seed)
    oracle = []
    for _, user in df_users.iterrows():
        eligible = df_cards[df_cards['income_requirement'] <= user['annual_income']]
        scores = sorted(
            [{'card_name': c['card_name'], 'net': calculate_net_benefit(user, c)}
             for _, c in eligible.iterrows()],
            key=lambda x: x['net'], reverse=True)
        top3 = scores[:3]
        if noise > 0 and rng.random() < noise:
            idx = rng.randint(1, 2)
            remaining = scores[3:]
            if remaining:
                top3[idx] = rng.choice(remaining)
        for rank, item in enumerate(top3, 1):
            oracle.append({'user_id': user['user_id'], 'card_name': item['card_name'], 'rank': rank})
    return pd.DataFrame(oracle)

def build_interaction_matrix(df_oracle, df_cards):
    df_o = df_oracle.copy()
    df_o['relevance'] = 4 - df_o['rank']  # rank1=3, rank2=2, rank3=1
    matrix = pd.pivot_table(df_o, index='user_id', columns='card_name',
                             values='relevance', fill_value=0)
    for card in df_cards['card_name']:
        if card not in matrix.columns: matrix[card] = 0
    return matrix[df_cards['card_name']]

oracle_clean = build_oracle(df_users, df_cards, noise=0.0, seed=42)
matrix_clean = build_interaction_matrix(oracle_clean, df_cards)
train_idx, test_idx = train_test_split(matrix_clean.index, test_size=0.2, random_state=42)
train_mat  = matrix_clean.loc[train_idx]
test_mat   = matrix_clean.loc[test_idx]
test_users = df_users[df_users['user_id'].isin(test_mat.index)].reset_index(drop=True)
card_names = df_cards['card_name'].tolist()
print(f'Train: {len(train_mat)} | Test: {len(test_mat)}')


## Models: UBCF, Cosine-CB, RAG-Hybrid, Oracle

In [ ]:
# --- UBCF ---
def run_ubcf(train, test):
    train_np, test_np = train.values, test.values
    preds = {}
    for i, uid in enumerate(test.index):
        sim = cosine_similarity(test_np[i].reshape(1,-1), train_np)[0]
        preds[uid] = np.dot(sim, train_np)
    return preds

# --- Cosine Content-Based ---
def run_cosine_cb(df_users_sub, df_cards):
    cats = ['travel','dining','fuel','shopping','online']
    card_mat = df_cards[[f'{c}_reward_rate' for c in cats]].values.astype(float)
    norms = np.linalg.norm(card_mat, axis=1, keepdims=True); norms[norms==0] = 1
    card_mat_n = card_mat / norms
    preds = {}
    for _, user in df_users_sub.iterrows():
        u_vec = np.array([user[f'{c}_spend'] for c in cats], dtype=float)
        n = np.linalg.norm(u_vec)
        if n > 0: u_vec /= n
        elig = np.array([1 if df_cards.iloc[j]['income_requirement'] <= user['annual_income'] else 0
                          for j in range(len(df_cards))], dtype=float)
        preds[user['user_id']] = (card_mat_n @ u_vec) * elig
    return preds

# --- RAG Hybrid ---
def build_card_documents(df_cards):
    docs = []
    for _, c in df_cards.iterrows():
        doc = (f"{c['card_name']} by {c['bank']}. Annual fee: Rs {c['annual_fee']}. "
               f"Income requirement: Rs {c['income_requirement']}. "
               f"Travel reward: {c['travel_reward_rate']}%. Dining: {c['dining_reward_rate']}%. "
               f"Fuel: {c['fuel_reward_rate']}%. Shopping: {c['shopping_reward_rate']}%. "
               f"Online: {c['online_reward_rate']}%. "
               f"Welcome bonus: Rs {c['welcome_bonus_value']}. "
               f"Lounge access: {c['lounge_access']} visits/year. "
               f"Forex markup: {c['forex_markup']}%.")
        docs.append(doc)
    return docs

def build_user_query(user):
    cats = ['travel','dining','fuel','shopping','online']
    top_cat = max(cats, key=lambda c: user[f'{c}_spend'])
    return (f"User with annual income Rs {int(user['annual_income'])} and monthly spending: "
            f"travel Rs {user['travel_spend']:.0f}, dining Rs {user['dining_spend']:.0f}, "
            f"fuel Rs {user['fuel_spend']:.0f}, shopping Rs {user['shopping_spend']:.0f}, "
            f"online Rs {user['online_spend']:.0f}. Primary category: {top_cat}. "
            f"Looking for a credit card that maximizes rewards.")

def build_rag_index(df_cards, model):
    docs = build_card_documents(df_cards)
    embeddings = model.encode(docs, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))
    return index, embeddings, docs

def run_rag_hybrid(df_users_sub, df_cards, model, index, alpha=0.5, k=8):
    """
    ScoreRAG(u,c) = alpha * sim_cos(q_u, d_c) + (1-alpha) * norm(NetBenefit(u,c))
    alpha=0 -> pure financial utility | alpha=1 -> pure semantic retrieval
    """
    n_cards = len(df_cards)
    nb_matrix = np.zeros((len(df_users_sub), n_cards))
    for i, (_, user) in enumerate(df_users_sub.iterrows()):
        for j, (_, card) in enumerate(df_cards.iterrows()):
            if card['income_requirement'] <= user['annual_income']:
                nb_matrix[i, j] = calculate_net_benefit(user, card)
    preds = {}
    for i, (_, user) in enumerate(df_users_sub.iterrows()):
        q_emb = model.encode([build_user_query(user)], normalize_embeddings=True).astype(np.float32)
        _, I = index.search(q_emb, k)
        sem_scores = np.zeros(n_cards)
        for rank, idx in enumerate(I[0]):
            sem_scores[idx] = 1.0 - rank/k
        nb = nb_matrix[i].copy()
        nb_norm = (nb - nb.min()) / (nb.max() - nb.min() + 1e-9)
        elig = np.array([1 if df_cards.iloc[j]['income_requirement'] <= user['annual_income'] else 0
                          for j in range(n_cards)])
        preds[user['user_id']] = (alpha * sem_scores + (1-alpha) * nb_norm) * elig
    return preds

def run_oracle_preds(df_users_sub, df_cards):
    preds = {}
    for _, user in df_users_sub.iterrows():
        scores = np.zeros(len(df_cards))
        for j, (_, card) in enumerate(df_cards.iterrows()):
            if card['income_requirement'] <= user['annual_income']:
                scores[j] = calculate_net_benefit(user, card)
        preds[user['user_id']] = scores
    return preds

# Load model
print('Loading SentenceTransformer (all-MiniLM-L6-v2)...')
model = SentenceTransformer('all-MiniLM-L6-v2')
t0 = time.time()
index, card_embeddings, card_docs = build_rag_index(df_cards, model)
print(f'FAISS index built in {time.time()-t0:.2f}s | Dimensions: {card_embeddings.shape[1]}')


## Evaluation Metrics: Precision@3, Hit Rate@3, NDCG@3

In [ ]:
def evaluate_preds(preds, test_df, card_names):
    p_list, hr_list, ndcg_list = [], [], []
    for uid in test_df.index:
        scores  = preds.get(uid, np.zeros(len(card_names)))
        top3    = pd.Series(scores, index=card_names).sort_values(ascending=False).head(3).index.tolist()
        true_vec = test_df.loc[uid]
        rel_dict = true_vec.to_dict()
        relevant = set(true_vec[true_vec > 0].index.tolist())
        p_list.append(len(set(top3) & relevant) / 3)
        hr_list.append(int(len(set(top3) & relevant) > 0))
        dcg  = sum(rel_dict.get(c,0)/np.log2(i+2) for i,c in enumerate(top3))
        idcg = sum(r/np.log2(i+2) for i,r in enumerate(sorted(rel_dict.values(),reverse=True)[:3]))
        ndcg_list.append(dcg/idcg if idcg>0 else 0)
    return (round(float(np.mean(p_list)),3),
            round(float(np.mean(hr_list)),3),
            round(float(np.mean(ndcg_list)),3),
            ndcg_list)


## Scenario 1 — Dense Deterministic

In [ ]:
t0=time.time(); preds_oracle=run_oracle_preds(test_users,df_cards); lat_oracle=(time.time()-t0)/len(test_users)*1000
t0=time.time(); preds_ubcf  =run_ubcf(train_mat,test_mat);           lat_ubcf  =(time.time()-t0)/len(test_users)*1000
t0=time.time(); preds_cos   =run_cosine_cb(test_users,df_cards);      lat_cos   =(time.time()-t0)/len(test_users)*1000
t0=time.time(); preds_rag   =run_rag_hybrid(test_users,df_cards,model,index,alpha=0.5); lat_rag=(time.time()-t0)/len(test_users)*1000

r1_oracle=evaluate_preds(preds_oracle,test_mat,card_names)
r1_ubcf  =evaluate_preds(preds_ubcf,  test_mat,card_names)
r1_cos   =evaluate_preds(preds_cos,   test_mat,card_names)
r1_rag   =evaluate_preds(preds_rag,   test_mat,card_names)

df_s1 = pd.DataFrame({
    'Model':        ['Oracle','UBCF','Cosine-CB','RAG-Hybrid (a=0.5)'],
    'Precision@3':  [r1_oracle[0],r1_ubcf[0],r1_cos[0],r1_rag[0]],
    'HitRate@3':    [r1_oracle[1],r1_ubcf[1],r1_cos[1],r1_rag[1]],
    'NDCG@3':       [r1_oracle[2],r1_ubcf[2],r1_cos[2],r1_rag[2]],
    'Latency(ms)':  [round(lat_oracle,2),round(lat_ubcf,2),round(lat_cos,2),round(lat_rag,2)]
})
print(df_s1.to_string(index=False))


## Scenarios 2–4: Noise, Weak Cold-Start, Strong Cold-Start

In [ ]:
# Scenario 2 — Behavioral Noise 30%
oracle_noisy = build_oracle(df_users, df_cards, noise=0.3, seed=123)
matrix_noisy = build_interaction_matrix(oracle_noisy, df_cards)
train_noisy  = matrix_noisy.loc[train_idx]; test_noisy = matrix_noisy.loc[test_idx]
r2_ubcf=evaluate_preds(run_ubcf(train_noisy,test_noisy),         test_noisy,card_names)
r2_cos =evaluate_preds(run_cosine_cb(test_users,df_cards),        test_noisy,card_names)
r2_rag =evaluate_preds(run_rag_hybrid(test_users,df_cards,model,index,alpha=0.5),test_noisy,card_names)

# Scenario 3 — Weak Cold-Start
WEAK_CARDS = ['SBI SimplySAVE','Axis Neo','ICICI Coral']
train_weak = train_mat.copy()
for c in WEAK_CARDS:
    if c in train_weak.columns: train_weak[c] = 0
r3_ubcf=evaluate_preds(run_ubcf(train_weak,test_mat),             test_mat,card_names)
r3_cos =evaluate_preds(run_cosine_cb(test_users,df_cards),         test_mat,card_names)
r3_rag =evaluate_preds(run_rag_hybrid(test_users,df_cards,model,index,alpha=0.5),test_mat,card_names)

# Scenario 4 — Strong Cold-Start
STRONG_CARDS = ['HDFC Millennia','ICICI Sapphiro','SBI PRIME','Axis Flipkart','HDFC Regalia Gold']
train_strong = train_mat.copy()
for c in STRONG_CARDS:
    if c in train_strong.columns: train_strong[c] = 0
r4_ubcf=evaluate_preds(run_ubcf(train_strong,test_mat),           test_mat,card_names)
r4_cos =evaluate_preds(run_cosine_cb(test_users,df_cards),         test_mat,card_names)
r4_rag =evaluate_preds(run_rag_hybrid(test_users,df_cards,model,index,alpha=0.5),test_mat,card_names)

# Summary table
df_ablation = pd.DataFrame({
    'Scenario':    ['Dense','Noise-30%','Weak CS','Strong CS'],
    'Oracle':      [r1_oracle[2],'—','—','—'],
    'UBCF':        [r1_ubcf[2], r2_ubcf[2], r3_ubcf[2], r4_ubcf[2]],
    'Cosine-CB':   [r1_cos[2],  r2_cos[2],  r3_cos[2],  r4_cos[2]],
    'RAG(a=0.5)':  [r1_rag[2],  r2_rag[2],  r3_rag[2],  r4_rag[2]],
    'UBCF-RAG Gap':[round(r1_ubcf[2]-r1_rag[2],3),round(r2_ubcf[2]-r2_rag[2],3),
                    round(r3_ubcf[2]-r3_rag[2],3),round(r4_ubcf[2]-r4_rag[2],3)],
})
print(df_ablation.to_string(index=False))


## Alpha Ablation Study

In [ ]:
alpha_rows = []
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    preds_a = run_rag_hybrid(test_users, df_cards, model, index, alpha=alpha)
    r_a = evaluate_preds(preds_a, test_mat, card_names)
    alpha_rows.append({'alpha': alpha, 'Precision@3': r_a[0], 'HitRate@3': r_a[1], 'NDCG@3': r_a[2]})

df_alpha = pd.DataFrame(alpha_rows)
print(df_alpha.to_string(index=False))
print('\nKey finding: alpha=0.0 (pure financial utility) achieves near-oracle NDCG@3.')
print('Recommended deployment: alpha=0.25 for cold-start scenarios.')


## Statistical Tests (Paired t-test, UBCF vs RAG-Hybrid)

In [ ]:
print('Paired t-test: UBCF vs RAG-Hybrid (alpha=0.5) on per-user NDCG@3')
for name, (u, r) in [('Dense',    (r1_ubcf[3],r1_rag[3])),
                      ('Noise-30%',(r2_ubcf[3],r2_rag[3])),
                      ('Weak CS',  (r3_ubcf[3],r3_rag[3])),
                      ('Strong CS',(r4_ubcf[3],r4_rag[3]))]:
    t, p = stats.ttest_rel(u, r)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*'
    print(f'  {name:12s}: t({len(u)-1}) = {t:7.3f},  p = {p:.4e}  {sig}')


## Case Study — Explainability Demo (alpha=0.25)

In [ ]:
# Representative user: mid-income, heavy shopping spender
sample_user = df_users[(df_users['annual_income'].between(700000,900000))].iloc[0]
cats = ['travel','dining','fuel','shopping','online']
print(f'User ID: {int(sample_user["user_id"])}  |  Annual Income: Rs {int(sample_user["annual_income"]):,}')
for c in cats: print(f'  {c:10s}: Rs {sample_user[f"{c}_spend"]:.0f}/month')

eligible = df_cards[df_cards['income_requirement'] <= sample_user['annual_income']]
nb_scores = {row['card_name']: calculate_net_benefit(sample_user, row) for _, row in eligible.iterrows()}
oracle_top3 = sorted(nb_scores.items(), key=lambda x: x[1], reverse=True)[:3]
print('\nOracle Top-3 (by NetBenefit):')
for i,(n,v) in enumerate(oracle_top3,1): print(f'  {i}. {n}: Net benefit Rs {v:,.0f}/year')

# RAG top-3 with alpha=0.25 (recommended setting)
preds_s = run_rag_hybrid(pd.DataFrame([sample_user]), df_cards, model, index, alpha=0.25)
uid_key = list(preds_s.keys())[0]
rag_top3 = pd.Series(preds_s[uid_key], index=card_names).sort_values(ascending=False).head(3).index.tolist()

def generate_explanation(user, card_name, df_cards, nb_scores):
    card = df_cards[df_cards['card_name']==card_name].iloc[0]
    top_cat = max(['travel','dining','fuel','shopping','online'], key=lambda c: user[f'{c}_spend'])
    annual_r = sum(user[f'{c}_spend']*12*(card[f'{c}_reward_rate']/100)
                   for c in ['travel','dining','fuel','shopping','online'])
    nb = nb_scores.get(card_name, 0)
    return (f"[{card_name} | {card['bank']}] Primary category: {top_cat} "
            f"(Rs {user[f'{top_cat}_spend']:.0f}/mo). "
            f"{top_cat.title()} reward rate: {card[f'{top_cat}_reward_rate']}%. "
            f"Est. annual reward: Rs {annual_r:,.0f}. "
            f"Annual fee: Rs {card['annual_fee']}. Net benefit: Rs {nb:,.0f}/year.")

print('\nRAG-Hybrid (alpha=0.25) Top-3 with Explanations:')
for i,cn in enumerate(rag_top3,1): print(f'  {i}. {generate_explanation(sample_user, cn, df_cards, nb_scores)}')
